# 🧩 实验 8：CartPole 上的 TD 方法

在前面的实验中，我们主要关注 **❄️ FrozenLake**。这是一个离散、低维的环境，有助于我们理解强化学习中的一些基本概念，例如 **马尔可夫决策过程（MDP）**、**状态转移** 以及 **表格型价值函数**。

现在，我们将从简单的网格世界进一步扩展，开始探索一个更真实的控制问题——**🏗️ CartPole 环境**。CartPole 是一个连续且动态变化的系统，被广泛用于强化学习研究中。

## 需要实现的算法

1. **蒙特卡洛方法（Monte Carlo, MC）**  
   通过完整的 episode 计算回报，并利用这些回报更新价值估计。

2. **SARSA（On-Policy TD Control）**  
   按照当前的 $\varepsilon$-greedy 策略与环境交互，并根据智能体实际执行的动作进行学习。

3. **Q-Learning（Off-Policy TD Control）**  
   在行为策略与目标策略可以不同的情况下学习最优策略，即学习过程不完全依赖智能体当前实际采用的行为策略。

## Part 1：理解 CartPole 环境

现在，我们将从静态的网格世界，转向一个**动态控制系统**——Gymnasium 中经典的 `CartPole-v1` 环境。

我们先创建这个环境，并随机执行几步动作，观察它的运行方式。

In [1]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML
from matplotlib import animation

# Use rgb_array so env.render() returns frames
env = gym.make("CartPole-v1", render_mode="rgb_array")
obs, info = env.reset(seed=0)

# Simple rule: push in the direction that reduces pole angle.
# If theta > 0 (pole tilts to the right), push right; else push left.
def greedy_policy(observation):
    _, _, theta, theta_dot = observation
    return 1 if theta > 0 else 0

frames = []
rewards = []
num_steps = 100

# capture initial frame
frames.append(env.render())

for t in range(num_steps):
    action = greedy_policy(obs)
    obs, r, terminated, truncated, info = env.step(action)
    rewards.append(r)
    frames.append(env.render())
    if terminated or truncated:
        # If episode ends early, reset and keep going until 100 frames collected
        obs, info = env.reset()
        frames.append(env.render())

env.close()

# Stack frames into a numpy array (T, H, W, C), dtype=uint8
frames = np.asarray(frames, dtype=np.uint8)
print(f"Frames shape: {frames.shape}, dtype: {frames.dtype}")

fig, ax = plt.subplots()
ax.set_axis_off()
img = ax.imshow(frames[0])

def animate(i):
    img.set_data(frames[i])
    return [img]

ani = animation.FuncAnimation(fig, animate, frames=len(frames), interval=30, blit=True)
plt.close(fig)  # prevent duplicate static display
HTML(ani.to_jshtml())

Frames shape: (103, 400, 600, 3), dtype: uint8


---

### 环境概览

**CartPole** 环境模拟的是一个经典控制问题：  
一根杆通过一个不受控的关节连接在小车上，小车可以在无摩擦轨道上左右移动。  
智能体需要对小车施加向左或向右的力，使杆尽可能保持竖直平衡。

**Episode 终止条件：**

- 杆的角度超过 **±12°**（约为 ±0.418 弧度）
- 小车的位置超过 **±2.4 米**
- Episode 长度达到 **500 个时间步**

在每一个时间步中，只要杆仍然保持平衡，智能体就会获得 **+1** 的奖励。

### 观测空间（状态空间）

环境返回的观测值是一个 4 维连续向量：

| 索引 | 变量 | 含义 | 典型范围 |
|:---:|---|---|---|
| 0 | `x` | 小车位置（m） | −2.4 ~ 2.4 |
| 1 | `x_dot` | 小车速度（m/s） | −∞ ~ ∞ |
| 2 | `theta` | 杆的角度（radians） | −0.418 ~ 0.418 |
| 3 | `theta_dot` | 杆的角速度（radians/s） | −∞ ~ ∞ |

这些连续状态值通常需要被**离散化**成若干区间，才能用于表格型强化学习算法，例如 **Monte Carlo**、**SARSA** 和 **Q-Learning**。

### 动作空间

| 动作 | 含义 |
|:---:|---|
| `0` | 将小车向**左**推 |
| `1` | 将小车向**右**推 |

在每个时间步：

1. 智能体观察当前状态 \(s_t\)。
2. 选择一个动作 \(a_t \in \{0, 1\}\)。
3. 如果杆仍然保持竖直平衡，则获得奖励 \(r_t = +1\)。

目标是**最大化累计总奖励**，也就是让杆尽可能长时间地保持平衡。

## Part 2：状态空间离散化

与 **FrozenLake** 这种离散的网格世界不同，**CartPole** 环境具有一个**连续状态空间**。每个状态由四个实数变量表示：

$$
s = [x, \dot{x}, \theta, \dot{\theta}]
$$

为了使用 **Monte Carlo**、**SARSA** 或 **Q-Learning** 这类**表格型方法**，我们需要先将连续状态转换为一种**离散表示**。

### 为什么需要离散化？

- 表格型强化学习算法需要用离散状态作为索引来访问 Q-table。
- 连续变量，例如杆的角度和速度，可以取无限多个值，因此不可能直接为每一个连续状态都存储一个表格项。
- 因此，我们会将每个状态维度划分为有限数量的**区间（bins）**，并把每一个观测值映射到对应的**区间索引（bin index）**。

In [ ]:
import numpy as np

n_actions = env.action_space.n
np.set_printoptions(precision=3, suppress=True)

# ----- Discretization -----
NUM_BINS = np.array([6, 6, 12, 12])
STATE_BOUNDS = np.array([
    [-2.4,   2.4],
    [-3.0,   3.0],
    [-0.418, 0.418],
    [-2.0,   2.0]
])

def discretize_state(obs):
    lo, hi = STATE_BOUNDS[:,0], STATE_BOUNDS[:,1]
    ratios = (np.clip(obs, lo, hi) - lo) / (hi - lo)
    bins = (ratios * NUM_BINS).astype(int)
    return tuple(np.clip(bins, 0, NUM_BINS - 1))

In [ ]:
# now print the new discrete state space
obs, _ = env.reset(seed=0)
discrete_state = discretize_state(obs)
print("Observation:", obs)
print("Discretized State:", discrete_state)

## Part 3：CartPole 上的蒙特卡洛控制

现在我们已经得到了**离散化后的状态空间**，因此可以使用 **Monte Carlo（MC）** 方法来估计最优控制策略。

蒙特卡洛方法直接从**完整的 episode** 中学习。它会等到一个 episode 结束之后，再根据每个访问过的状态-动作对之后获得的**总回报**来更新 Q 值。

---

### 核心思想

对于每一个 episode：

1. 生成一条由状态、动作和奖励组成的轨迹：

   $$
   (s_0, a_0, r_1, s_1, a_1, r_2, \ldots, s_T)
   $$

2. 计算回报：

   $$
   G_t = r_{t+1} + \gamma r_{t+2} + \cdots + \gamma^{T-t-1} r_T
   $$

3. 更新 Q 值：

   $$
   Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha \big(G_t - Q(s_t, a_t)\big)
   $$

4. 使用 $\varepsilon$-greedy 探索策略得到动作选择方式：

   $$
   \pi(s) = 
   \begin{cases}
   \arg\max_a Q(s, a), & \text{利用，即选择当前 Q 值最大的动作} \\
   \text{随机动作}, & \text{探索，即以概率 } \varepsilon \text{ 随机选择动作}
   \end{cases}
   $$

---

In [ ]:
# ----- ε-greedy policy -----
def greedy_action(Q, s, epsilon):
    if np.random.rand() < epsilon:
        return np.random.randint(n_actions)
    return int(np.argmax(Q[s]))

In [ ]:
# ----- Initialize tables -----
Q = np.zeros((*NUM_BINS, n_actions))
N = np.zeros((*NUM_BINS, n_actions), dtype=int)


num_episodes = 5000
gamma = 0.99
epsilon = 1.0
eps_min, eps_decay = 0.05, 0.995
returns_history = []


for ep in range(1, num_episodes + 1):
    obs, _ = env.reset()
    episode = []

    # Your time to work on it 
    done = False
    while not done:
        s = discretize_state(obs)
        a = #    
        obs_next, r, term, trunc, _ = env.step(a)
        episode.append((s, a, r))
        obs = obs_next
        done = term or trunc

    T = len(episode)
    G = 0.0
    returns = np.zeros(T)
    for t in range(T - 1, -1, -1):
        _, _, r = episode[t]
        G = gamma * G + r
        returns[t] = G

    for t, (s, a, _) in enumerate(episode):
        
        N[s][a] = #
        n =  #
        Q[s][a] = #


    epsilon = max(eps_min, epsilon * eps_decay)
    ep_return = sum(r for _, _, r in episode)
    returns_history.append(ep_return)

    if ep % 500 == 0:
        avg = np.mean(returns_history[-100:])
        print(f"Episode {ep:5d} | ε={epsilon:.3f} | AvgReturn(100)={avg:.1f}")

In [ ]:
def evaluate_greedy(Q, episodes=20):
    total = 0.0
    for _ in range(episodes):
        obs, _ = env.reset()
        done = False
        ep_ret = 0.0
        while not done:
            s = discretize_state(obs)
            a = np.argmax(Q[s])
            obs, r, term, trunc, _ = env.step(a)
            ep_ret += r
            done = term or trunc
        total += ep_ret
    return total / episodes

avg_eval = evaluate_greedy(Q)
print(f"\nGreedy policy average return: {avg_eval:.1f}")

## Part 4：n-Step SARSA（On-Policy TD Control）

**目标：** 将 SARSA 从 1-step target 扩展到 **n-step return**，在偏差和方差之间进行折中。相比纯 Monte Carlo 方法，n-step SARSA 可以实现更平滑、更稳定的学习。

### 核心思想

对于每一个时间索引 $\tau$，我们构造一个 **n-step target**。它会累计未来最多 $n$ 步的奖励；如果 episode 还没有结束，则从 $S_{\tau+n}, A_{\tau+n}$ 处的 $Q$ 值进行 **bootstrapping**：

$$
G_{\tau:\tau+n} \;=\; \sum_{i=\tau+1}^{\min(\tau+n,\,T)} \gamma^{\,i-(\tau+1)}\,R_i \;+\; 
\begin{cases}
\gamma^{\,n}\, Q(S_{\tau+n}, A_{\tau+n}) & \text{如果 } \tau+n < T,\\
0 & \text{否则。}
\end{cases}
$$

更新公式为：

$$
Q(S_\tau, A_\tau) \leftarrow Q(S_\tau, A_\tau)\;+\;\alpha\left[\,G_{\tau:\tau+n} - Q(S_\tau, A_\tau)\,\right].
$$

### 实现机制：缓存与索引

- 维护滚动缓存：**状态** $S_0,S_1,\dots$，**动作** $A_0,A_1,\dots$，以及 **奖励** $R_1,R_2,\dots$。
- 令 $T$ 表示 episode 终止时的时间步，也就是第一次出现终止或截断的时间步。
- 在每一步 $t$，一旦 $\tau = t-n+1 \ge 0$，就可以计算 $(S_\tau,A_\tau)$ 对应的 n-step target。
- 当 $\tau = T-1$ 时停止。

### 伪代码（高层描述）

1. 重置环境；使用 $\varepsilon$-greedy 策略选择 $A_0$。
2. 对 $t=0,1,\dots$ 重复：
   - 执行动作 $A_t$，观察 $R_{t+1}, S_{t+1}$；如果不是终止状态，则用 $\varepsilon$-greedy 策略选择 $A_{t+1}$。
   - 令 $\tau = t - n + 1$。  
     如果 $\tau \ge 0$：  
     &nbsp;&nbsp;• 计算 $G_{\tau:\tau+n}$，即最多 $n$ 步奖励之和，加上必要时的 bootstrap 项。  
     &nbsp;&nbsp;• 将 $Q(S_\tau,A_\tau)$ 向 $G_{\tau:\tau+n}$ 的方向更新。
   - 当 $\tau = T-1$ 时停止。

In [ ]:
# ----- Tables & hyperparams -----
Q = np.zeros((*NUM_BINS, n_actions))
num_episodes = 4000
max_steps = 1000
gamma = 0.99
alpha = 0.1
n = 4                           # n-step horizon (tune me)
epsilon = 1.0
eps_min, eps_decay = 0.05, 0.995

returns_hist = []

for ep in range(1, num_episodes + 1):
    obs, _ = env.reset()
    s0 = discretize_state(obs)
    a0 = greedy_action(Q, s0, epsilon)

    # Buffers (index from 0); r[t] stores r_{t}, so shift by 1 for clarity with algorithm
    S = [s0]            # states S_0, S_1, ...
    A = [a0]            # actions A_0, A_1, ...
    R = [0.0]           # R[0] unused; will append R_1, R_2, ...
    T = np.inf

    t = 0
    ep_return = 0.0

    while True:
        if t < T:
            # Step in env using A_t
            obs_next, r, term, trunc, _ = env.step(A[t])
            ep_return += r
            done = bool(term or trunc)
            R.append(r)                       # this is R_{t+1}
            if done:
                T = t + 1
            else:
                s_next = discretize_state(obs_next)
                S.append(s_next)              # S_{t+1}
                a_next = greedy_action(Q, s_next, epsilon)
                A.append(a_next)              # A_{t+1}

        tau = t - n + 1                       # state to update
        if tau >= 0:
            # Compute G (n-step return starting at tau)
            # G = sum_{i=tau+1}^{min(tau+n, T)} gamma^{i-(tau+1)} R_i

            # Your time to work on it 
            G = 0.0
            upper = int(min(tau + n, T))
            power = 0
            for i in range(tau + 1, upper + 1):
                G = ####
                power = #### 
            if tau + n < T:                   # bootstrap if within episode
                
                G = ######

            s_tau = S[tau]
            a_tau = A[tau]
            Q[s_tau][a_tau]= ##### 

        if tau == T - 1:
            break
        t += 1

    # ε schedule & logging
    epsilon = max(eps_min, epsilon * eps_decay)
    returns_hist.append(ep_return)
    if ep % 500 == 0:
        avg = np.mean(returns_hist[-100:])
        print(f"Episode {ep:5d} | ε={epsilon:.3f} | AvgReturn(100)={avg:.1f}")

In [ ]:
avg_eval = evaluate_greedy(Q, episodes=20)
print(f"\nGreedy policy average return (n={n} SARSA): {avg_eval:.1f}")

## Part 5：One-Step Q-Learning（Off-Policy TD Control）

### 核心更新规则

对于每一个转移样本 $ (s_t, a_t, r_{t+1}, s_{t+1}) $：

$$
Q(s_t, a_t) \;\leftarrow\; Q(s_t, a_t) \;+\;
\alpha \,\Big[\, r_{t+1} \;+\; \gamma \max_{a'} Q(s_{t+1}, a') \;-\; Q(s_t, a_t) \,\Big].
$$

- **Target** 项 $ r_{t+1} + \gamma \max_{a'} Q(s_{t+1}, a') $ 使用的是当前 $Q$ 估计下的**最优下一步动作**。
- 因此，Q-Learning 即使在执行一个带有探索性的 $\varepsilon$-greedy 行为策略时，也可以学习到一个**最优策略**。

In [ ]:
Q = np.zeros((*NUM_BINS, n_actions), dtype=float)
num_episodes = 4000
gamma = 0.99
alpha = 0.1
epsilon = 1.0
eps_min, eps_decay = 0.05, 0.995
returns_hist = []

for ep in range(1, num_episodes + 1):
    obs, _ = env.reset()
    done = False
    ep_return = 0.0

    while not done:
        s = discretize_state(obs)
        a = greedy_action(Q, s, 1)  # Use random exploration as greedy action

        obs_next, r, term, trunc, _ = env.step(a)
        done = bool(term or trunc)
        ep_return += r

        s_next = discretize_state(obs_next)
        
        # ........ Your time to work on it ........
        if done: 
            td_target = r
        else:
            td_target = #
            
        Q[s][a] = #

        obs = obs_next

    # ε schedule & logging
    epsilon = max(eps_min, epsilon * eps_decay)
    returns_hist.append(ep_return)
    if ep % 500 == 0:
        avg = np.mean(returns_hist[-100:])
        print(f"Episode {ep:5d} | ε={epsilon:.3f} | AvgReturn(100)={avg:.1f}")

In [ ]:
avg_eval = evaluate_greedy(Q, episodes=20)
print(f"\nGreedy policy average return (1-step Q-learning): {avg_eval:.1f}")

## 🧭 Part 6：创建我们自己的环境

我们将实现一个简单的 **3×3 GridWorld**，用来学习如何构建自定义的 Gym 环境。

- **起点：** \((0,0)\)，**目标：** \((2,2)\)
- **动作：** \(0=\text{上},\,1=\text{下},\,2=\text{左},\,3=\text{右}\)
- **奖励：** 到达目标时获得 \(+1\)，每走一步获得 \(-0.01\)，撞墙时获得 \(-0.05\)
- **终止条件：** 到达目标，或者达到最大步数限制

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

# ----------------------------
# 1) Custom 3x3 Grid Environment
# ----------------------------
class Grid3x3Env(gym.Env):
    """
    A tiny 3x3 GridWorld.
      - Start at (0,0), goal at (2,2)
      - Actions: 0=Up, 1=Down, 2=Left, 3=Right
      - Rewards: +1 on reaching goal, -0.01 per step, -0.05 for bumping into a wall (stay in place)
      - Episode ends on goal or step limit
    Observation: Discrete(9) index r*3 + c
    """
    metadata = {"render_modes": ["ansi"]}

    def __init__(self, render_mode=None, max_steps=30):
        super().__init__()
        self.N = 3
        self.action_space = spaces.Discrete(4)
        self.observation_space = spaces.Discrete(self.N * self.N)

        self.start = (0, 0)
        self.goal  = (2, 2)
        self.max_steps = max_steps
        self.render_mode = render_mode

        self._pos = None
        self._steps = 0

        # (dr, dc) for Up, Down, Left, Right
        self._moves = [(-1,0), (1,0), (0,-1), (0,1)]

    def _rc_to_obs(self, r, c): return r * self.N + c
    def _obs_to_rc(self, obs):  return divmod(obs, self.N)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._pos = self.start
        self._steps = 0
        return self._rc_to_obs(*self._pos), {}

    def step(self, action):
        self._steps += 1
        r, c = self._pos
        dr, dc = self._moves[int(action)]
        nr, nc = r + dr, c + dc

        reward = -0.01
        terminated = False
        truncated = self._steps >= self.max_steps

        # check bounds
        if 0 <= nr < self.N and 0 <= nc < self.N:
            self._pos = (nr, nc)
        else:
            # bump wall: stay & extra penalty
            reward -= 0.05

        if self._pos == self.goal:
            reward = 1.0
            terminated = True

        return self._rc_to_obs(*self._pos), reward, terminated, truncated, {}

    def render(self):
        board = [[" . "]*self.N for _ in range(self.N)]
        gr, gc = self.goal
        board[gr][gc] = "[G]"
        r, c = self._pos
        board[r][c] = " A "
        return "\n".join("".join(row) for row in board)

In [ ]:
# ----------------------------
# Random trajectory demo
# ----------------------------
import time

env = Grid3x3Env()
obs, _ = env.reset()
done = False

print("Initial state:")
print(env.render(), "\n")

for t in range(10):  # up to 10 random steps
    if done:
        break
    a = env.action_space.sample()
    obs, r, term, trunc, _ = env.step(a)
    done = term or trunc
    print(f"Step {t+1}, Action={a}, Reward={r:.2f}")
    print(env.render(), "\n")
    time.sleep(0.5)

if done:
    print("Episode ended.")
else:
    print("Trajectory finished without termination.")